In [ ]:
"""
Sliding-window chunker.

Splits page text into fixed-size chunks using tiktoken cl100k_base.
Chunk size: 400 tokens, overlap: 80 tokens (locked in config.yaml).
Floor: 50 tokens — discard fragments below this.
Ceiling: 512 tokens — reranker constraint. Assert fires after heading injection.

Heading prefix (from pdf_loader.py Tier 2 injection on table pages) is prepended
to every chunk from that page.
"""

import json
import logging
from pathlib import Path

import tiktoken

from ingestion.pdf_loader import DOC_REGISTRY, load_pdf

logger = logging.getLogger(__name__)

_tokenizer = tiktoken.get_encoding("cl100k_base")

CHUNK_SIZE = 400
OVERLAP = 80
MIN_TOKENS = 50
MAX_TOKENS = 512

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


In [2]:
def chunk_page(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = OVERLAP) -> list[str]:
    """Split text into token-bounded chunks with overlap. Discards fragments below MIN_TOKENS."""
    tokens = _tokenizer.encode(text)
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + chunk_size, len(tokens))
        chunk_tokens = tokens[start:end]
        if len(chunk_tokens) >= MIN_TOKENS:
            chunks.append(_tokenizer.decode(chunk_tokens))
        start += chunk_size - overlap
    return chunks

In [3]:
def chunk_document(path: str | Path, output_dir: str | Path) -> list[dict]:
    """
    Load a PDF, apply sliding-window chunking, validate, save to output_dir.
    Returns the list of chunk dicts saved.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    pages = load_pdf(path)
    if not pages:
        logger.warning("No pages extracted from %s", path)
        return []

    all_chunks = []
    chunk_idx = 0

    for page in pages:
        heading_prefix = page.get("heading_prefix", "")
        meta = {k: v for k, v in page.items() if k != "heading_prefix"}

        for chunk_text in chunk_page(page["text"]):
            final_text = f"{heading_prefix}\n{chunk_text}" if heading_prefix else chunk_text
            token_count = len(_tokenizer.encode(final_text))
            assert token_count <= MAX_TOKENS, (
                f"Chunk exceeds {MAX_TOKENS}-token ceiling after heading injection: "
                f"{token_count} tokens in {meta['doc_id']} page {meta['page_number']}"
            )
            all_chunks.append({
                **meta,
                "text": final_text,
                "chunk_index": chunk_idx,
                "token_count": token_count,
            })
            chunk_idx += 1

    if not all_chunks:
        logger.warning("No chunks produced for %s", pages[0]["doc_id"])
        return []

    doc_id = pages[0]["doc_id"]
    out_path = output_dir / f"{doc_id}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(all_chunks, f, ensure_ascii=False, indent=2)

    avg_tokens = sum(c["token_count"] for c in all_chunks) / len(all_chunks)
    logger.info(
        "Chunked %s: %d chunks, avg %.0f tokens, max %d tokens",
        doc_id, len(all_chunks), avg_tokens,
        max(c["token_count"] for c in all_chunks),
    )
    return all_chunks


In [4]:
def chunk_all(raw_dir: str | Path, output_dir: str | Path) -> dict:
    """Chunk all registered PDFs in raw_dir. Returns summary stats per doc."""
    raw_dir = Path(raw_dir)
    stats = {}

    for filename, meta in DOC_REGISTRY.items():
        pdf_path = raw_dir / filename
        if not pdf_path.exists():
            logger.warning("PDF not found, skipping: %s", pdf_path)
            continue
        chunks = chunk_document(pdf_path, output_dir)
        if chunks:
            stats[meta["doc_id"]] = {
                "chunk_count": len(chunks),
                "avg_tokens": sum(c["token_count"] for c in chunks) / len(chunks),
                "max_tokens": max(c["token_count"] for c in chunks),
            }

    total = sum(s["chunk_count"] for s in stats.values())
    logger.info("Total chunks across all documents: %d", total)
    return stats

In [5]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
output_dir = ROOT / "data" / "processed"

stats = {}
for f in sorted(output_dir.glob("*.json")):
    chunks = json.loads(f.read_text(encoding="utf-8"))
    if not chunks:
        continue
    stats[f.stem] = {
        "chunk_count": len(chunks),
        "avg_tokens": sum(c["token_count"] for c in chunks) / len(chunks),
        "max_tokens": max(c["token_count"] for c in chunks),
        "table_chunks": sum(1 for c in chunks if c.get("chunk_type") == "table"),
    }

total = sum(s["chunk_count"] for s in stats.values())
print(f"Total chunks across {len(stats)} documents: {total}\n")
for doc_id, s in stats.items():
    print(f"{doc_id:35s} {s['chunk_count']:4d} chunks | avg {s['avg_tokens']:.0f} tok | max {s['max_tokens']:4d} tok | {s['table_chunks']} table chunks")

assert all(s["max_tokens"] <= MAX_TOKENS for s in stats.values()), "A doc exceeds the 512-token ceiling"
stats

Total chunks across 12 documents: 3402

BOE_CBES_KEY_ELEMENTS_2021            85 chunks | avg 260 tok | max  400 tok | 16 table chunks
BOE_CBES_RESULTS_2021                126 chunks | avg 279 tok | max  400 tok | 15 table chunks
BOE_DISCLOSURE_2024                  103 chunks | avg 306 tok | max  423 tok | 18 table chunks
BOE_MACRO_IMPLICATIONS                55 chunks | avg 286 tok | max  400 tok | 0 table chunks
BOE_MEASURING_CLIMATE_RISKS           57 chunks | avg 280 tok | max  400 tok | 5 table chunks
CCC_PROGRESS_2024                    201 chunks | avg 307 tok | max  421 tok | 48 table chunks
CCC_PROGRESS_2025                    286 chunks | avg 314 tok | max  422 tok | 41 table chunks
CCC_SEVENTH_CARBON_BUDGET_2025       884 chunks | avg 317 tok | max  430 tok | 71 table chunks
DESNZ_ZEV_MANDATE_2024                69 chunks | avg 254 tok | max  400 tok | 9 table chunks
ESO_BEYOND2030_2024                  272 chunks | avg 283 tok | max  400 tok | 74 table chunks
IEA_WEO_2025 

{'BOE_CBES_KEY_ELEMENTS_2021': {'chunk_count': 85,
  'avg_tokens': 260.45882352941175,
  'max_tokens': 400,
  'table_chunks': 16},
 'BOE_CBES_RESULTS_2021': {'chunk_count': 126,
  'avg_tokens': 278.55555555555554,
  'max_tokens': 400,
  'table_chunks': 15},
 'BOE_DISCLOSURE_2024': {'chunk_count': 103,
  'avg_tokens': 306.2135922330097,
  'max_tokens': 423,
  'table_chunks': 18},
 'BOE_MACRO_IMPLICATIONS': {'chunk_count': 55,
  'avg_tokens': 286.03636363636366,
  'max_tokens': 400,
  'table_chunks': 0},
 'BOE_MEASURING_CLIMATE_RISKS': {'chunk_count': 57,
  'avg_tokens': 279.82456140350877,
  'max_tokens': 400,
  'table_chunks': 5},
 'CCC_PROGRESS_2024': {'chunk_count': 201,
  'avg_tokens': 306.99004975124376,
  'max_tokens': 421,
  'table_chunks': 48},
 'CCC_PROGRESS_2025': {'chunk_count': 286,
  'avg_tokens': 313.64335664335664,
  'max_tokens': 422,
  'table_chunks': 41},
 'CCC_SEVENTH_CARBON_BUDGET_2025': {'chunk_count': 884,
  'avg_tokens': 316.89027149321265,
  'max_tokens': 430,
  